# A Tale of Two Grids: Forecasting Electricity in Texas vs. New York

**Lucas Azenha & Anya Tu — Columbia University, 2026**

Can a seasonal ARIMA model, paired with a GARCH volatility model, forecast daily statewide electricity in Texas and New York? And what does the difference in forecast accuracy say about the two grids?

**Outline**
1. Setup
2. Data from the U.S. Energy Information Administration (EIA) API
3. Stationarity check and model selection
4. SARIMA-GARCH model for each state
5. Extension: adding weather (cooling and heating degree days) for New York
6. Results

## 1. Setup

In [ ]:
# Uncomment on a fresh environment (for example Google Colab):
# !pip install -q statsmodels arch pmdarima

In [ ]:
import os
from getpass import getpass

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from arch import arch_model

plt.rcParams["figure.figsize"] = [12, 6]

The EIA API needs a free key ([register here](https://www.eia.gov/opendata/register.php)). The key is read from the `EIA_API_KEY` environment variable so it never appears in the notebook.

In [ ]:
EIA_API_KEY = os.environ.get("EIA_API_KEY") or getpass("EIA API key: ")

In [ ]:
# Model settings
ORDER = (1, 0, 2)               # non-seasonal (p, d, q)
SEASONAL_ORDER = (0, 1, 1, 7)   # seasonal (P, D, Q, s): weekly seasonality
TRAIN_FRACTION = 0.8

# False reproduces the original analysis. True applies three corrections
# (see "Data notes" below): keep one timezone copy of each day, drop the first
# and last day (which can be incomplete), and leave the SARIMA start-up
# residuals out of the GARCH fit.
APPLY_FIXES = False

## 2. Data

Both series come from the EIA Open Data API (`electricity/rto`), at daily frequency:

- **Texas (ERCOT):** `daily-region-sub-ba-data`, summed over the East, Far West, South Central, South, and West subregions
- **New York (NYISO):** `daily-fuel-type-data`, summed over fuel types

In [ ]:
EIA_BASE = "https://api.eia.gov/v2/electricity/rto"
PAGE_SIZE = 5000   # the API returns at most 5,000 rows per request


def fetch_eia(route, facet, values, start, end):
    """Download the most recent rows of a daily EIA series as a DataFrame."""
    facets = "".join(f"&facets[{facet}][]={v}" for v in values)
    url = (
        f"{EIA_BASE}/{route}/data/?frequency=daily&data[0]=value{facets}"
        f"&start={start}&end={end}"
        f"&sort[0][column]=period&sort[0][direction]=desc"
        f"&offset=0&length={PAGE_SIZE}&api_key={EIA_API_KEY}"
    )
    payload = requests.get(url, timeout=60).json()
    if "response" not in payload:
        raise RuntimeError(f"EIA request failed: {payload.get('error', payload)}")
    return pd.DataFrame(payload["response"]["data"])


def to_daily_series(raw, strict=None):
    """Turn raw EIA rows into one clean daily series (MWh)."""
    strict = APPLY_FIXES if strict is None else strict
    df = raw.copy()
    df["period"] = pd.to_datetime(df["period"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["value"])
    if strict and "timezone" in df.columns:
        df = df[df["timezone"] == df["timezone"].iloc[0]]
    series = df.groupby("period")["value"].sum().sort_index()
    if strict:
        series = series.iloc[1:-1]
    return series.asfreq("D").interpolate()

In [ ]:
texas_raw = fetch_eia(
    "daily-region-sub-ba-data", "subba",
    ["EAST", "FWES", "SCEN", "SOUT", "WEST"],
    start="2019-01-01", end="2026-06-20",
)
new_york_raw = fetch_eia(
    "daily-fuel-type-data", "respondent",
    ["NY", "NYIS"],
    start="2025-01-01", end="2026-06-29",
)

texas = to_daily_series(texas_raw)
new_york = to_daily_series(new_york_raw)

texas_raw.head()

In [ ]:
for name, raw, series in [("Texas", texas_raw, texas), ("New York", new_york_raw, new_york)]:
    print(f"{name}: {len(raw):,} raw rows, {len(series)} days "
          f"({series.index.min().date()} to {series.index.max().date()}), "
          f"{raw['value'].isna().sum()} missing values")
    print(f"  Highest day: {series.idxmax().date()}")

**Data notes**

- The API returns at most 5,000 rows per request, so each series covers only the most recent days in the requested range. The date ranges printed above are the actual analysis windows.
- The API returns each day once per timezone. With `APPLY_FIXES = False` these copies are summed together, which scales the level of the series.
- The first and last day of each window can be incomplete, and the first week of SARIMA residuals are start-up values from the seasonal differencing, not real forecast errors. With `APPLY_FIXES = False` both are left in, as in the original analysis.
- Set `APPLY_FIXES = True` to correct all three. The results will change.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
texas.plot(ax=axes[0], title="Texas (ERCOT): daily electricity")
new_york.plot(ax=axes[1], title="New York (NYISO): daily electricity")
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("MWh")
    ax.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

## 3. Stationarity check and model selection

Each series is log-transformed and split 80/20 into train and test sets. An Augmented Dickey-Fuller (ADF) test checks stationarity of the training data, and `auto_arima` searches for a seasonal order with a weekly period.

In [ ]:
def split_log(series, fraction=TRAIN_FRACTION):
    """Log-transform a series and split it into train and test sets."""
    log_series = np.log(series)
    split = int(len(log_series) * fraction)
    return log_series[:split], log_series[split:]


for name, series in [("Texas", texas), ("New York", new_york)]:
    train, _ = split_log(series)
    print(f"{name}: ADF p-value = {adfuller(train.dropna())[1]:.4f}")

In [ ]:
import pmdarima as pm

for name, series in [("Texas", texas), ("New York", new_york)]:
    print(f"\n=== {name} ===")
    auto_model = pm.auto_arima(
        series, seasonal=True, m=7, d=0, D=1, max_p=2, max_q=2,
        trace=True, error_action="ignore", suppress_warnings=True,
    )
    print(auto_model.summary())

## 4. SARIMA-GARCH model

The model has two layers:

1. **SARIMA** `(1,0,2)×(0,1,1)₇` on the log series forecasts the mean.
2. **GARCH(1,1)** with skewed Student's t errors is fit on the SARIMA residuals to model time-varying volatility. It supplies the 95% band around the forecast.

Residual checks: Ljung-Box tests for leftover autocorrelation, and the ARCH-LM test checks whether a GARCH layer is justified.

In [ ]:
def run_sarima_garch(series, name, exog=None):
    """Fit SARIMA(X) + GARCH(1,1), forecast the test window, and plot the results."""
    train, test = split_log(series)
    split = len(train)
    exog_train = exog.iloc[:split] if exog is not None else None
    exog_test = exog.iloc[split:] if exog is not None else None
    label = "SARIMAX" if exog is not None else "SARIMA"

    # Mean model
    sarima_fit = SARIMAX(
        train, exog=exog_train, order=ORDER, seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False, enforce_invertibility=False,
    ).fit(disp=False)
    print(f"=== {name}: {label} summary ===")
    print(sarima_fit.summary())

    # Residual diagnostics
    resid = sarima_fit.resid.dropna()
    if APPLY_FIXES:
        resid = resid.iloc[SEASONAL_ORDER[3] + 1:]   # drop start-up residuals
    print("\nLjung-Box on residuals (want p > 0.05):")
    print(acorr_ljungbox(resid, lags=[7, 14], return_df=True))
    arch_lm_p = het_arch(resid, nlags=7)[1]
    print(f"ARCH-LM test p-value: {arch_lm_p:.4f}")

    # Volatility model (residuals scaled by 100 for numerical stability)
    garch_fit = arch_model(resid * 100, vol="Garch", p=1, q=1, dist="skewt").fit(disp="off")
    print(f"\n=== {name}: GARCH summary ===")
    print(garch_fit.summary())

    # Forecast the test window and undo the log transform
    forecast_log = sarima_fit.forecast(steps=len(test), exog=exog_test)
    forecast_log.index = test.index
    forecast = np.exp(forecast_log)
    actual = np.exp(test)

    variance = garch_fit.forecast(horizon=len(test), reindex=False).variance.values.flatten() / 100**2
    std_dev = np.sqrt(variance)
    lower = np.exp(forecast_log - 1.96 * std_dev)
    upper = np.exp(forecast_log + 1.96 * std_dev)

    errors = actual - forecast
    metrics = {
        "Model": f"{label}-GARCH",
        "Train days": len(train),
        "Test days": len(test),
        "MAPE (%)": np.mean(np.abs(errors / actual)) * 100,
        "RMSE (MWh)": np.sqrt(np.mean(errors**2)),
        "MAE (MWh)": np.mean(np.abs(errors)),
    }
    print(f"\n{name} {label}-GARCH MAPE: {metrics['MAPE (%)']:.2f}%")

    plt.figure()
    plt.plot(np.exp(train), label="Train", color="gray", alpha=0.5)
    plt.plot(actual, label="Actual (test)", color="tab:blue", linewidth=1.5)
    plt.plot(forecast, label=f"{label} forecast", color="tab:red", linestyle="--")
    plt.fill_between(actual.index, lower, upper, color="tab:pink", alpha=0.3, label="GARCH 95% volatility band")
    plt.title(f"{name}: {label}-GARCH forecast")
    plt.xlabel("Date")
    plt.ylabel("Daily electricity (MWh)")
    plt.legend(loc="upper left")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.show()

    volatility = garch_fit.conditional_volatility / 100
    plt.figure(figsize=(12, 4))
    plt.plot(volatility.index, volatility, color="tab:orange", linewidth=1.2)
    plt.title(f"{name}: GARCH conditional volatility (training period)")
    plt.xlabel("Date")
    plt.ylabel("Conditional volatility")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.show()

    return metrics


results = {}

### Texas

In [ ]:
results["Texas"] = run_sarima_garch(texas, "Texas")

### New York

In [ ]:
results["New York"] = run_sarima_garch(new_york, "New York")

## 5. Extension: adding weather for New York

Temperature drives electricity use, so this section adds two weather inputs to the New York model: cooling degree days (CDD) and heating degree days (HDD), built from daily maximum temperature in New York City ([Open-Meteo archive API](https://open-meteo.com/), no key needed).

**Look-ahead caveat:** the test period uses the weather that actually occurred, not a weather forecast. The error below is therefore the model's skill given perfect weather information, which is better than a real forecast could achieve.

In [ ]:
def fetch_degree_days(start, end, latitude=40.7128, longitude=-74.0060, base_temp_f=65):
    """Daily cooling and heating degree days from Open-Meteo's weather archive."""
    payload = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": latitude, "longitude": longitude,
            "start_date": start, "end_date": end,
            "daily": "temperature_2m_max",
            "temperature_unit": "fahrenheit",
            "timezone": "America/New_York",
        },
        timeout=60,
    ).json()
    temp = pd.Series(
        payload["daily"]["temperature_2m_max"],
        index=pd.to_datetime(payload["daily"]["time"]),
        name="temp_f",
    )
    return pd.DataFrame({
        "cdd": (temp - base_temp_f).clip(lower=0),
        "hdd": (base_temp_f - temp).clip(lower=0),
    })


weather = fetch_degree_days(
    new_york.index.min().strftime("%Y-%m-%d"),
    new_york.index.max().strftime("%Y-%m-%d"),
)
ny_weather = (
    pd.concat([new_york.rename("value"), weather], axis=1, join="inner")
    .asfreq("D")
    .interpolate()
)

results["New York + weather"] = run_sarima_garch(
    ny_weather["value"], "New York", exog=ny_weather[["cdd", "hdd"]]
)

## 6. Results

In [ ]:
summary = pd.DataFrame(results).T
summary.style.format({
    "MAPE (%)": "{:.2f}", "RMSE (MWh)": "{:,.0f}", "MAE (MWh)": "{:,.0f}",
})

**Takeaways**

- New York's forecast error is much lower than Texas's.
- Texas residuals are strongly heavy-tailed: the model misses occasional large shocks such as Winter Storm Fern (January 2026), which a linear SARIMA structure cannot anticipate.
- The GARCH layer does not change the mean forecast. Its role is to model those shocks as time-varying volatility and to give an honest uncertainty band.

**Limitations and next steps**

- Short analysis windows, set by the API's 5,000-row limit. Paging through the full history would allow longer training periods and yearly seasonality.
- A single train/test split. Rolling-origin evaluation would give a more reliable error estimate.
- Compare against machine learning baselines (gradient boosting, LSTM) and use forecast weather instead of observed weather.